# 02. Модуль 2: распознавание эмоций

Представления аудио (сигнал, лог-мел, HuBERT, латенты VAE) и функции потерь (CE, взвешенная CE, focal). Главная метрика: macro F1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!test -d /content/call-audio-analysis || git clone -q https://github.com/tanyalebedeva/call-audio-analysis.git /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
REPORTS = '/content/drive/MyDrive/coursework/reports'
!mkdir -p $DATA $RUNS $REPORTS
!nvidia-smi -L

## Представления аудио, функция потерь CE

In [ ]:
!python -m callaudio.train --config configs/emotion/mel_cnn.yaml

In [ ]:
!python -m callaudio.train --config configs/emotion/wave_cnn.yaml

In [ ]:
!python -m callaudio.train --config configs/emotion/hubert_frozen.yaml

In [ ]:
!python -m callaudio.train --config configs/emotion/hubert_finetune.yaml

Классификатор на латентах VAE требует обученной модели `vae_dusha` (ноутбук 03).

In [ ]:
!python -m callaudio.train --config configs/emotion/latent_vae.yaml

## Функции потерь (MelCNN)

In [ ]:
for loss in ['weighted_ce', 'focal']:
    !python -m callaudio.train --config configs/emotion/mel_cnn.yaml --set loss.name={loss} out_dir={RUNS}/emo_mel_{loss}

## Оценка: чистая речь, шум 5 дБ, шум и очистка модулем 1

In [ ]:
import glob
ENHANCER = f'{RUNS}/sep_unet_sisdr/best.pt'
for ckpt in sorted(glob.glob(f'{RUNS}/emo_*/best.pt')):
    !python -m callaudio.evaluate emotion --ckpt {ckpt}
    !python -m callaudio.evaluate emotion --ckpt {ckpt} --noise-snr 5
    !python -m callaudio.evaluate emotion --ckpt {ckpt} --noise-snr 5 --enhancer {ENHANCER}

In [ ]:
!python -m callaudio.report --runs $RUNS --out $REPORTS
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{REPORTS}/emotion.md', encoding='utf-8').read()))
display(Image(f'{RUNS}/emo_mel_ce/confusion.png'))

## Веса слоёв HuBERT

In [ ]:
import json, os
import matplotlib.pyplot as plt
for run in ['emo_hubert_frozen_ce', 'emo_hubert_ft_ce']:
    p = f'{RUNS}/{run}/results_emotion.json'
    if os.path.exists(p):
        plt.plot(json.load(open(p))['layer_importance'], marker='o', label=run)
plt.xlabel('слой'); plt.ylabel('вес'); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f'{REPORTS}/hubert_layers.png', dpi=150); plt.show()

## Пространство эмбеддингов по эпохам

In [ ]:
!python -m callaudio.viz --run {RUNS}/emo_mel_ce
from IPython.display import Image
Image(f'{RUNS}/emo_mel_ce/viz/umap_evolution.gif')